In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler


In [18]:
display(df.columns)

Index(['dst_port', 'protocol', 'duration', 'total_bytes', 'total_packets',
       'fwd_pkts', 'fwd_bytes', 'fwd_pkt_len_mean', 'fwd_pkt_len_std',
       'fwd_pkt_len_min', 'fwd_pkt_len_max', 'fwd_iat_mean', 'fwd_iat_std',
       'fwd_iat_min', 'fwd_iat_max', 'fwd_syn', 'fwd_ack', 'fwd_fin',
       'fwd_rst', 'fwd_psh', 'fwd_urg', 'bwd_pkts', 'bwd_bytes',
       'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'bwd_pkt_len_min',
       'bwd_pkt_len_max', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_min',
       'bwd_iat_max', 'bwd_syn', 'bwd_ack', 'bwd_fin', 'bwd_rst', 'bwd_psh',
       'bwd_urg', 'bytes_per_second', 'packets_per_second',
       'fwd_bytes_per_second', 'bwd_bytes_per_second', 'down_up_ratio',
       'flow_pkt_asymmetry', 'flow_byte_asymmetry', 'pkt_len_mean',
       'pkt_len_std', 'pkt_len_min', 'pkt_len_max', 'pkt_len_q1', 'pkt_len_q3',
       'pkt_len_skew', 'pkt_len_entropy', 'small_pkt_ratio', 'large_pkt_ratio',
       'iat_mean', 'iat_std', 'iat_min', 'iat_max', 'burstiness',
   

In [3]:
df = pd.read_csv(r"C:\Users\lenovo\OneDrive\Desktop\minor project\data.csv")
print(f"Dataset Shape: {df.shape}")
print("\nFirst 10 rows:")
display(df.head(20))

Dataset Shape: (28132, 89)

First 10 rows:


,dst_port,protocol,duration,total_bytes,total_packets,fwd_pkts,fwd_bytes,fwd_pkt_len_mean,fwd_pkt_len_std,fwd_pkt_len_min,...,cipher_suite,dns_query_count,dns_query_types,dns_unique_queries,http_request_count,http_unique_hosts,http_unique_uris,label,fingerprint_final,fingerprint_method
0,443,TCP,0.414215,4391,18,9,1069,118.7778,96.0599,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
1,443,TCP,0.058407,4369,17,9,1101,122.3333,98.8849,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
2,443,TCP,0.038559,4261,15,8,1047,130.8750,101.7048,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
3,443,TCP,0.037885,4261,15,8,1047,130.8750,101.7048,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
4,443,TCP,0.037970,4261,15,8,1047,130.8750,101.7048,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
5,443,TCP,0.038854,4369,17,9,1101,122.3333,98.8849,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
6,443,TCP,0.038668,4369,17,9,1101,122.3333,98.8849,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
7,443,TCP,0.038317,4369,17,9,1101,122.3333,98.8849,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
8,443,TCP,0.039744,4261,15,8,1047,130.8750,101.7048,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4
9,443,TCP,0.037553,4369,17,9,1101,122.3333,98.8849,54.0,...,0x3c,0,0,0,0,0,0,minor_project,t12i210300_b973bfd88a0e_c08b0bbc99a6,ja4


#  Check missing values and labels

In [4]:
print("--- Missing Values Per Critical Feature ---")
critical_cols = ['ja4_fingerprint', 'ja3_fingerprint', 'has_tls', 'label']
print(df[critical_cols].isnull().sum())

--- Missing Values Per Critical Feature ---
ja4_fingerprint    0
ja3_fingerprint    0
has_tls            0
label              0
dtype: int64


In [5]:
print("\n--- Distribution of Labels ---")
print(df['label'].value_counts())


--- Distribution of Labels ---
label
emotet           13977
trickbot         10904
minor_project     3192
dreambot            59
Name: count, dtype: int64


# Clean data, filter out non-fingerprinted flows, and handle infinities

1. Filter out traffic where no fingerprint or final fingerprint is missing

In [6]:
df_cleaned = df.dropna(subset=['fingerprint_final']).copy()
print(f"Row count after removing missing fingerprints: {len(df_cleaned)}")

Row count after removing missing fingerprints: 28132


In [7]:


# 2. Handle numerical columns containing potential Inf / -Inf from rate computations
# (e.g., 'bytes_per_second', 'packets_per_second')
num_cols = df_cleaned.select_dtypes(include=[np.number]).columns
df_cleaned[num_cols] = df_cleaned[num_cols].replace([np.inf, -np.inf], np.nan)

# Fill leftover NaNs in numeric attributes with their column medians
for col in num_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# 3. Discard highly unique or completely static tracking columns that cause overfitting
cols_to_drop = [c for c in df_cleaned.columns if df_cleaned[c].nunique() <= 1]
df_cleaned = df_cleaned.drop(columns=cols_to_drop)
print(f"Final cleaned dataset shape: {df_cleaned.shape}")

Final cleaned dataset shape: (28132, 73)


#  Categorical text/fingerprint field mapping

In [8]:
# Identify object/string type features to transform safely
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns.tolist()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_18068\326160934.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_cleaned.select_dtypes(include=['object']).columns.tolist()


In [9]:
if 'label' in categorical_cols:
    categorical_cols.remove('label')

print(f"Encoding categorical features: {categorical_cols}")

Encoding categorical features: ['ja4_fingerprint', 'ja3_fingerprint', 'cipher_suite', 'fingerprint_final']


Categorical Encoding

In [10]:
encoders = {}
for col in categorical_cols:
    # Convert all values to string type first to safely bypass mixed data types
    df_cleaned[col] = df_cleaned[col].astype(str)
    le = LabelEncoder()
    df_cleaned[col] = le.fit_transform(df_cleaned[col])
    encoders[col] = le


# Create Open-Set / Closed-Set split configurations

In [11]:
# Encode your target class labels
target_encoder = LabelEncoder()
df_cleaned['label_encoded'] = target_encoder.fit_transform(df_cleaned['label'])
print("Original Classes:", target_encoder.classes_)

Original Classes: ['dreambot' 'emotet' 'minor_project' 'trickbot']



 KNOWN_CLASSES = ['benign', 'emotet', 'trickbot'], UNKNOWN_CLASS = 'zeus'

In [12]:
all_classes = list(target_encoder.classes_)
UNKNOWN_CLASS = all_classes[-1] # Grabbing the last class as an evaluation dummy placeholder
KNOWN_CLASSES = [c for c in all_classes if c != UNKNOWN_CLASS]

In [13]:
print(f"\nTraining/Validation Classes (Closed Set): {KNOWN_CLASSES}")
print(f"Withheld Target for Rejection Testing (Open Set Unknown): ['{UNKNOWN_CLASS}']")


Training/Validation Classes (Closed Set): ['dreambot', 'emotet', 'minor_project']
Withheld Target for Rejection Testing (Open Set Unknown): ['trickbot']


In [14]:
print(f"\nTraining/Validation Classes (Closed Set): {KNOWN_CLASSES}")
print(f"Withheld Target for Rejection Testing (Open Set Unknown): ['{UNKNOWN_CLASS}']")

# Separate data into Known and Unknown partitions
df_known = df_cleaned[df_cleaned['label'].isin(KNOWN_CLASSES)].copy()
df_unknown = df_cleaned[df_cleaned['label'] == UNKNOWN_CLASS].copy()

# Drop raw target string identifiers to prepare feature matrices
X_known = df_known.drop(columns=['label', 'label_encoded'])
y_known = df_known['label_encoded']

X_unknown_test = df_unknown.drop(columns=['label', 'label_encoded'])
y_unknown_test = df_unknown['label_encoded']

# Split the known data into 80% Train, 10% Validation, and 10% Closed-Set Test
X_train, X_temp, y_train, y_temp = train_test_split(X_known, y_known, test_size=0.20, random_state=42, stratify=y_known)
X_val, X_close_test, y_val, y_close_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\nData Splits Setup Successfully:")
print(f" -> Train shapes: {X_train.shape}, Labels distribution:\n{y_train.value_counts()}")
print(f" -> Val shape: {X_val.shape}")
print(f" -> Closed-Set Test shape: {X_close_test.shape}")
print(f" -> Open-Set Unknown Test shape: {X_unknown_test.shape}")


Training/Validation Classes (Closed Set): ['dreambot', 'emotet', 'minor_project']
Withheld Target for Rejection Testing (Open Set Unknown): ['trickbot']

Data Splits Setup Successfully:
 -> Train shapes: (13782, 72), Labels distribution:
label_encoded
1    11181
2     2554
0       47
Name: count, dtype: int64
 -> Val shape: (1723, 72)
 -> Closed-Set Test shape: (1723, 72)
 -> Open-Set Unknown Test shape: (10904, 72)


#  Fit scaler exclusively to training features

In [15]:

scaler = MinMaxScaler()

# Scaling features matrices
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_close_test_scaled = scaler.transform(X_close_test)
X_unknown_test_scaled = scaler.transform(X_unknown_test)


# 1D CNN MODEL: , 
upto know needs to be rechecked?